In [1]:
import numpy as np
import pandas as pd
import torch
import os
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer,
    TrainingArguments, 
    Trainer, 
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

c:\Users\alexy\miniconda3\envs\vacuna_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# import gc
# import torch

# del model
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

NameError: name 'model' is not defined

In [2]:
TRAIN_FILE = "LoRA_training20260316.xlsx"
TEST_FILE = "LoRA_testing20260316.xlsx"
# MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 
# OUTPUT_DIR = "./my_final_lora_model"
MODEL_ID = "lmsys/vicuna-7b-v1.5" 
OUTPUT_DIR = "./vicuna_final_lora_model"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f">>> Currently using device: {device}")

>>> Currently using device: cuda


In [3]:
df_train = pd.read_excel(TRAIN_FILE)
print(df_train['再購'].value_counts())
print(df_train['再購'].value_counts(normalize=True))

再購
0    66
1    31
Name: count, dtype: int64
再購
0    0.680412
1    0.319588
Name: proportion, dtype: float64


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

vicuna_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "{{ message['content'] + ' ' }}"
    "{% elif message['role'] == 'user' %}"
    "{{ 'USER: ' + message['content'] + ' ' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ 'ASSISTANT: ' + message['content'] + '</s>' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ 'ASSISTANT:' }}"
    "{% endif %}"
)
tokenizer.chat_template = vicuna_template

In [5]:
df_train.head()

,Name,B,B*pathco,I,I_reverse,I_reverse*pathco,C,C*pathco,trust,202210~202212再購,再購
0,方雪櫻,6,1.296,0.0086,0.0364,0.019692,15,8.445,9.760692,0,0
1,王忠禮,2,0.432,0.0086,0.0364,0.019692,15,8.445,8.896692,0,0
2,安雅玫,5,1.080,0.0090,0.0360,0.019476,15,8.445,9.544476,0,0
3,朱芳綾,1,0.216,0.0073,0.0377,0.020396,15,8.445,8.681396,0,0
4,江琇枝,3,0.648,0.0073,0.0377,0.020396,15,8.445,9.113396,1,1


In [6]:
def process_data_to_chat(file_path, tokenizer):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Cannot find file: {file_path}")
    
    df = pd.read_excel(file_path)
    formatted_data = []
    
    for _, row in df.iterrows():
        # 在 User Prompt 中明確定義 1 與 0 的含義
        chat = [
            {
                "role": "system",
                "content": "You are a professional purchase behavior predictor and an expert in trust research."
            },
            {
                "role": "user", 
                "content": (
                    f"Trust is a multi-dimensional construct composed of Benevolence, Integrity, and Competence. "
                    f"Benevolence reflects goodwill and care toward customers (e.g., proactive communication and support). "
                    # f"Integrity reflects honesty and transparency. "
                    f"Competence reflects the ability and expertise to deliver expected service, often associated with experience and performance. "
                    f"Analyze the following values: "
                    f"Benevolence: {row['B']}, "
                    # f"Integrity: {row['I_reverse']}, "
                    f"Competence: {row['C']}. "
                    f"Predict if this user will purchase again. "
                    f"Output '1' if the user will buy again, or '0' if they will not. "
                    f"Answer with ONLY the number (0 or 1)."
                )
            },
            {"role": "assistant", "content": str(int(row['再購'])).strip()}
        ]
        text = tokenizer.apply_chat_template(chat, tokenize=False)  # 修正 typo: apply_chast -> apply_chat # 20260325修
        formatted_data.append({"text": text, "label": str(int(row['再購'])).strip()})
        
    return Dataset.from_list(formatted_data)

In [7]:
from datasets import concatenate_datasets

print(">>> Preparing datasets with defined labels (1=Purchase, 0=No Purchase)...")
full_train_dataset = process_data_to_chat(TRAIN_FILE, tokenizer)
dataset_split = full_train_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
val_dataset   = dataset_split["test"]

# 過採樣少數類（label=1），讓訓練集平衡 # 20260325修
# 原本 0:1 約 2:1，模型學到「都猜 0 loss 最小」導致 recall(1)=0
label_0 = train_dataset.filter(lambda x: x["label"] == "0")
label_1 = train_dataset.filter(lambda x: x["label"] == "1")
repeat_times = len(label_0) // len(label_1)  # 重複少數類讓數量接近
label_1_upsampled = concatenate_datasets([label_1] * repeat_times)
train_dataset = concatenate_datasets([label_0, label_1_upsampled]).shuffle(seed=42)

print(f"  label 0: {train_dataset['label'].count('0')}, label 1: {train_dataset['label'].count('1')}")

test_dataset_raw = process_data_to_chat(TEST_FILE, tokenizer)

>>> Preparing datasets with defined labels (1=Purchase, 0=No Purchase)...


Filter: 100%|██████████| 87/87 [00:00<00:00, 14467.13 examples/s]

  label 0: 59, label 1: 56


In [8]:
print(full_train_dataset[0]['text'])

You are a professional purchase behavior predictor and an expert in trust research. USER: Trust is a multi-dimensional construct composed of Benevolence, Integrity, and Competence. Benevolence reflects goodwill and care toward customers (e.g., proactive communication and support). Competence reflects the ability and expertise to deliver expected service, often associated with experience and performance. Analyze the following values: Benevolence: 6, Competence: 15. Predict if this user will purchase again. Output '1' if the user will buy again, or '0' if they will not. Answer with ONLY the number (0 or 1). ASSISTANT: 0</s>


In [9]:
print(test_dataset_raw['text'])
print(test_dataset_raw['label'])

Column(["You are a professional purchase behavior predictor and an expert in trust research. USER: Trust is a multi-dimensional construct composed of Benevolence, Integrity, and Competence. Benevolence reflects goodwill and care toward customers (e.g., proactive communication and support). Competence reflects the ability and expertise to deliver expected service, often associated with experience and performance. Analyze the following values: Benevolence: 2, Competence: 15. Predict if this user will purchase again. Output '1' if the user will buy again, or '0' if they will not. Answer with ONLY the number (0 or 1). ASSISTANT: 0</s>", "You are a professional purchase behavior predictor and an expert in trust research. USER: Trust is a multi-dimensional construct composed of Benevolence, Integrity, and Competence. Benevolence reflects goodwill and care toward customers (e.g., proactive communication and support). Competence reflects the ability and expertise to deliver expected service, o

In [10]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text", "label"])
tokenized_val = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text", "label"])

Map: 100%|██████████| 10/10 [00:00<00:00, 1999.19 examples/s]


In [12]:
import torch
import transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

torch: 2.5.1+cu121
transformers: 5.4.0


In [16]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    quantization_config=bnb_config,
    device_map="auto",
    use_safetensors=True
)

lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1, # 增加 Dropout 防止過擬合
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

# 關鍵：把可訓練參數轉成 fp32
# for param in model.parameters():
#     if param.requires_grad:
#         param.data = param.data.float()

# model.config.use_cache = False
# model.gradient_checkpointing_enable()

Loading weights: 100%|██████████| 291/291 [01:51<00:00,  2.60it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [17]:
training_args = TrainingArguments(
    output_dir="./lora_checkpoints_0325",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,  # 3e-4
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=8, 
    num_train_epochs=10,        
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

In [18]:
print("\n>>> Starting Fine-tuning...")
trainer.train()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


>>> Starting Fine-tuning...


Epoch,Training Loss,Validation Loss
1,No log,0.418786
2,No log,0.175826
3,No log,0.156621
4,No log,0.080919
5,No log,0.038766
6,No log,0.025622
7,No log,0.023231
8,No log,0.023348
9,No log,0.023039
10,No log,0.023620


('./vicuna_final_lora_model\\tokenizer_config.json',
 './vicuna_final_lora_model\\chat_template.jinja',
 './vicuna_final_lora_model\\tokenizer.json')

In [19]:
print("\n>>> Running blind test inference...")
model.eval()
y_true, y_pred = [], []

for i, item in enumerate(tqdm(test_dataset_raw, desc="Processing Samples")):
    prompt = item["text"].split("ASSISTANT:")[0] + "ASSISTANT:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    prediction_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    if i < 10:
        print(f"[{i}] raw output: {repr(prediction_text)} | label: {item['label']}")
    try:
        prediction = prediction_text[0] if prediction_text[0] in ['0', '1'] else "0"
    except:
        prediction = "0"
    y_true.append(item["label"])
    y_pred.append(prediction)


>>> Running blind test inference...


Processing Samples:   1%|          | 1/97 [00:00<01:29,  1.07it/s]Both `max_new_tokens` (=2) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[0] raw output: '0' | label: 0


Processing Samples:   2%|▏         | 2/97 [00:01<00:56,  1.69it/s]Both `max_new_tokens` (=2) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1] raw output: '1' | label: 0


...

Processing Samples:   9%|▉         | 9/97 [00:03<00:32,  2.67it/s]Both `max_new_tokens` (=2) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[8] raw output: '0' | label: 0


Processing Samples:  10%|█         | 10/97 [00:04<00:32,  2.67it/s]Both `max_new_tokens` (=2) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[9] raw output: '0' | label: 0


Processing Samples: 100%|██████████| 97/97 [00:35<00:00,  2.74it/s]


In [20]:
print("\n>>> Running blind test inference...")
model.eval()
y_true, y_pred = [], []

for item in tqdm(test_dataset_raw, desc="Processing Samples"):
    prompt = item["text"].split("ASSISTANT:")[0] + "ASSISTANT:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=2, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    prediction_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    try:
        prediction = prediction_text[0] if prediction_text[0] in ['0', '1'] else "0"
    except:
        prediction = "0"
    
    y_true.append(item["label"])
    y_pred.append(prediction)


>>> Running blind test inference...


Processing Samples:   0%|          | 0/97 [00:00<?, ?it/s]

Both `max_new_tokens` (=2) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Processing Samples: 100%|██████████| 97/97 [00:33<00:00,  2.86it/s]


In [21]:
print("\n" + "="*40)
print(f"Final Accuracy: {accuracy_score(y_true, y_pred):.2%}")
print("Label Definition: 1 = Will Purchase, 0 = Will Not Purchase")
print("="*40)
print(classification_report(y_true, y_pred, target_names=["Not Purchase (0)", "Purchase (1)"]))

# 匯出結果
pd.DataFrame({"Actual": y_true, "Predict": y_pred}).to_excel("final_report_v2.xlsx", index=False)
print(">>> Report generated: final_report_v2.xlsx")


Final Accuracy: 63.92%
Label Definition: 1 = Will Purchase, 0 = Will Not Purchase
                  precision    recall  f1-score   support

Not Purchase (0)       0.65      0.69      0.67        51
    Purchase (1)       0.63      0.59      0.61        46

        accuracy                           0.64        97
       macro avg       0.64      0.64      0.64        97
    weighted avg       0.64      0.64      0.64        97

>>> Report generated: final_report_v2.xlsx
